# GPU


## 预备
首先，确保你至少安装了一个NVIDIA GPU。
然后，下载[NVIDIA驱动和CUDA](https://developer.nvidia.com/cuda-downloads)
并按照提示设置适当的路径。
当这些准备工作完成，就可以使用`nvidia-smi`命令来(**查看显卡信息。**)

In [2]:
! nvidia-smi

Sun Oct 23 18:01:42 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.76       Driver Version: 515.76       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  Off  | 00000000:01:00.0 Off |                  N/A |
|  0%   38C    P8     5W / 250W |      5MiB / 11264MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  Off  | 00000000:03:00.0 Off |                  N/A |
|  0%   

在PyTorch中，每个数组都有一个设备（device），
我们通常将其称为上下文（context）。

例如，当在带有GPU的服务器上训练神经网络时，
我们通常希望模型的参数在GPU上。

在GPU上创建的张量只消耗这个GPU的显存。我们可以使用`nvidia-smi`命令查看显存使用情况。一般来说，我们需要确保不创建超过GPU显存限制的数据。

要运行此部分中的程序，至少需要两个GPU。


In [3]:
import torch
from torch import nn

## 计算设备变量


我们可以指定用于存储和计算的设备，如CPU和GPU。

默认情况下，所有变量和相关的计算都**分配给CPU**. `cpu`设备意味着所有物理CPU和内存，这意味着PyTorch的计算将尝试使用所有CPU核心。

然而，`gpu`设备只代表一个卡和相应的显存。如果有多个GPU，我们使用`torch.device(f'cuda:{i}')`来表示第$i$块GPU（$i$从0开始）。

另外，`cuda:0`和`cuda`是等价的。


In [25]:
# 分配CPU
d1 = torch.device('cpu')

# 分配第一块GPU
d2 = torch.device('cuda')

# 分配第二块GPU
d3 = torch.device('cuda:1')

# 另一种写法
d4 = torch.device('cuda', 0)

d1, d2, d3, d4

(device(type='cpu'),
 device(type='cuda'),
 device(type='cuda', index=1),
 device(type='cuda', index=0))

### 查询可用gpu的数量


In [26]:
torch.cuda.device_count()

2

### 自适应

> 一种写法

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

> 定义了两个方便的函数

In [6]:
# 如果存在，则返回某块gpu, 默认第一块, 否则返回cpu
def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

# 返回所有可用的GPU，如果没有GPU，则返回[cpu(),]
def try_all_gpus():
    devices = [torch.device(f'cuda:{i}')
               for i in range(torch.cuda.device_count())]
    if(devices == None):
        devices = [torch.device('cpu')]
    return devices


try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0), device(type='cuda', index=1)])

## 在GPU上存储张量

### 分配到GPU上

#### 创建张量时指定存储设备

In [32]:
# 我们可以在创建张量时指定存储设备
device = torch.device("cuda:0")
X = torch.ones(2, 3, device=device)
X, X.device

(tensor([[1., 1., 1.],
         [1., 1., 1.]], device='cuda:0'),
 device(type='cuda', index=0))

#### 移动到GPU上

In [20]:
Y = torch.rand(2, 3)
Y = Y.to(device=try_gpu(1))
Y, Y.device

(tensor([[0.1844, 0.5498, 0.0874],
         [0.9487, 0.0182, 0.1561]], device='cuda:1'),
 device(type='cuda', index=1))

### 复制



需要注意的是，无论何时我们要对多个项进行操作，它们都必须在同一个设备上。

例如，如果我们对两个张量求和，我们需要确保两个张量都位于同一个设备上，否则框架将不知道在哪里存储结果，甚至不知道在哪里执行计算。


*不要*简单地`X`加上`Y`，因为这会导致异常，运行时引擎不知道该怎么做：它在同一设备上找不到数据会导致失败。

由于`Y`位于第二个GPU上，所以我们需要将`X`移到那里，
然后才能执行相加运算。

![复制数据以在同一设备上执行操作](../img/copyto.svg)

In [21]:
# 复制X到第二块GPU上
Z = X.cuda(1)
print(X)
print(Z)
print(Y)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:1')
tensor([[0.1844, 0.5498, 0.0874],
        [0.9487, 0.0182, 0.1561]], device='cuda:1')


[**现在数据在同一个GPU上（`Z`和`Y`都在），我们可以将它们相加。**]


In [22]:
Y + Z

tensor([[1.1844, 1.5498, 1.0874],
        [1.9487, 1.0182, 1.1561]], device='cuda:1')

假设变量`Z`已经存在于第二个GPU上。
如果我们还是调用`Z.cuda(1)`会发生什么？
它将返回`Z`，而不会复制并分配新内存。


In [23]:
Z.cuda(1) is Z

True

人们使用GPU来进行机器学习，因为单个GPU相对运行速度快。

但是在设备（CPU、GPU和其他机器）之间传输数据比计算慢得多。这也使得并行化变得更加困难，因为我们必须等待数据被发送（或者接收），然后才能继续进行更多的操作。

这就是为什么拷贝操作要格外小心。根据经验，多个小操作比一个大操作糟糕得多。一次执行几个操作比代码中散布的许多单个操作要好得多（除非你确信自己在做什么）。

如果一个设备必须等待另一个设备才能执行其他操作，那么这样的操作可能会阻塞。这有点像排队订购咖啡，而不像通过电话预先订购：当你到店的时候，咖啡已经准备好了。

最后，当我们打印张量或将张量转换为NumPy格式时，如果数据不在内存中，框架会首先将其复制到内存中，这会导致额外的传输开销。更糟糕的是，它现在受制于全局解释器锁，使得一切都得等待Python完成。

### 查询张量所在的设备

默认情况下，张量是在CPU上创建的。


In [29]:
x = torch.tensor([1, 2, 3])
x, x.device

(tensor([1, 2, 3]), device(type='cpu'))


## 在GPU上存储神经网络

### 分配到GPU上

In [39]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

总之，只要所有的数据和参数都在同一个设备上，我们就可以有效地学习模型。

In [40]:
# `X`为GPU上的张量时，模型将在同一GPU上计算结果。
X = torch.ones(2, 3, device=try_gpu())
net(X)

tensor([[-0.0533],
        [-0.0533]], device='cuda:0', grad_fn=<AddmmBackward0>)

### 查询模型参数所在的设备


In [42]:
net[0].weight.data.device

device(type='cuda', index=0)